# MoE routing deep dive: balance, capacity and collapse
**LLMs, Animated · AR.1 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/LLM-Learner/blob/main/course/notebooks/ar-1-moe-routing.ipynb)

The lesson took the router apart. Here you rebuild the pieces in NumPy and then watch a router collapse, and recover, on a toy task:

1. Top-k routing with renormalized gates
2. The Switch load-balancing loss and the router z-loss
3. Expert capacity: which assignments get dropped
4. Training a tiny MoE with and without the balancing loss (you write its gradient)
5. DeepSeek-V3-style bias balancing, with no loss at all

Each exercise has a `# TODO` stub followed by a check cell. Solutions are at the end. Everything runs on a CPU in well under a minute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

## 1. Top-k routing with renormalized gates

For each token the router gives logits $z$ (one per expert). Keep the top $k$ and turn their probabilities into gates that sum to 1:

$$p = \mathrm{softmax}(z), \qquad g_i = \frac{p_i}{\sum_{j \in \mathrm{TopK}(p)} p_j}$$

Write `route(logits, k)` for a batch of logits of shape (T, E). Return `probs` (T, E), `idx` (T, k) with expert indices from best to worst, and `gates` (T, k).

In [ ]:
def route(logits, k):
    probs = softmax(logits)
    # TODO: idx = the k largest probabilities per row, best first (np.argsort helps)
    # TODO: gates = the chosen probabilities, renormalized to sum to 1 per row
    raise NotImplementedError

In [ ]:
# The lesson's worked example: logits (2.0, 1.0, 0.5, -1.0)
probs, idx, gates = route(np.array([[2.0, 1.0, 0.5, -1.0]]), k=2)
assert np.allclose(probs, [[0.6095, 0.2242, 0.1360, 0.0303]], atol=1e-3)
assert idx.tolist() == [[0, 1]]
assert np.allclose(gates, [[0.7311, 0.2689]], atol=1e-3)
# "top-k, then softmax over the kept logits" gives the same gates
assert np.allclose(gates[0, 0], np.exp(2) / (np.exp(2) + np.exp(1)))
rng = np.random.default_rng(0)
P, I, G = route(rng.normal(size=(50, 8)), k=3)
assert I.shape == (50, 3) and np.allclose(G.sum(1), 1) and (I[:, 0] == P.argmax(1)).all()
print("Exercise 1 passed:", gates)

## 2. The balancing loss and the z-loss

Switch Transformer's auxiliary loss, with $f_i$ the share of the $kT$ assignments that went to expert $i$ and $P_i$ the mean router probability for expert $i$:

$$\mathcal{L}_{\text{aux}} = E \sum_{i=1}^{E} f_i P_i \quad(\text{times a small } \alpha\text{, left out here})$$

ST-MoE's router z-loss, averaged over tokens:

$$\mathcal{L}_z = \frac{1}{T}\sum_{t}\Big(\log\sum_j e^{z_{t,j}}\Big)^2$$

Write both. Use a numerically stable log-sum-exp (subtract the row max first).

In [ ]:
def aux_loss(probs, idx):
    T, E = probs.shape
    # TODO: f = assignment counts per expert / idx.size ; P = probs.mean(0)
    raise NotImplementedError

def z_loss(logits):
    # TODO: stable log-sum-exp per row, squared, averaged
    raise NotImplementedError

In [ ]:
# Worked example from the lesson: 8 tokens, counts (4, 2, 1, 1), mean probs (0.4, 0.3, 0.15, 0.15)
P_ex = np.tile([0.4, 0.3, 0.15, 0.15], (8, 1))
idx_ex = np.array([[0], [0], [0], [0], [1], [1], [2], [3]])
assert np.isclose(aux_loss(P_ex, idx_ex), 1.25)
assert np.isclose(aux_loss(np.full((8, 4), 0.25), np.array([[0], [1], [2], [3]] * 2)), 1.0)
assert np.isclose(z_loss(np.array([[2.0, 1.0, 0.0, -1.0]])), 5.9545, atol=1e-3)
assert np.isclose(z_loss(np.array([[20.0, 10.0, 0.0, -10.0]])), 400.0018, atol=1e-2)
assert np.isfinite(z_loss(np.array([[1000.0, 0.0]]))), "use a stable log-sum-exp"
print("Exercise 2 passed")

## 3. Capacity: which assignments get dropped?

Each expert accepts at most $C = \lceil \text{cf} \times kT / E \rceil$ assignments per batch. Go through the tokens **in order** (token 0 first, and within a token its best expert first) and accept an assignment only if that expert still has room. Return a boolean `keep` array shaped like `idx` and the capacity `C`.

In [ ]:
def apply_capacity(idx, E, cf):
    T, k = idx.shape
    C = int(np.ceil(cf * k * T / E))
    keep = np.zeros_like(idx, dtype=bool)
    # TODO: count how many assignments each expert has accepted so far,
    #       and set keep[t, j] = True only while that count is below C
    return keep, C

In [ ]:
idx_c = np.array([[0], [0], [0], [1], [0], [2]])        # 6 tokens, top-1, 4 experts
keep, C = apply_capacity(idx_c, E=4, cf=1.0)            # C = ceil(6/4) = 2
assert C == 2
assert keep[:, 0].tolist() == [True, True, False, True, False, True]
# The lesson's numbers: T = 1024, k = 2, E = 8, cf = 1.25 -> C = 320
idx_big = np.zeros((1024, 2), dtype=int); idx_big[:200, 0] = 3; idx_big[:200, 1] = 3; idx_big[200:, 0] = 1; idx_big[200:, 1] = 2
keep_big, C_big = apply_capacity(idx_big, E=8, cf=1.25)
assert C_big == 320 and (~keep_big).sum() == 80 + 504 + 504
print("Exercise 3 passed: dropped", int((~keep_big).sum()), "of", keep_big.size)

## 4. Train a tiny MoE, with and without balancing

The toy task: inputs come from 8 clusters, and each cluster has its own linear map to the target. An MoE layer with 8 linear experts and **top-1** routing could give every cluster its own expert, if the router spreads tokens out. We start the router with a strong favourite (a bias of +4 on expert 0) to see whether training recovers.

The forward pass is $\hat y = p_{i^*}\,W_{i^*} x$. The training loop below computes the gradients for the MSE; your job is the extra gradient from $\alpha \cdot E \sum_i f_i P_i$. Since $f_i$ is a hard count (no gradient) and $P_i = \frac{1}{n}\sum_t p_{t,i}$, each token's probability $p_{t,i}$ gets an extra gradient of $\alpha E f_i / n$.

In [ ]:
C_, D_, O_, E_ = 8, 8, 4, 8
_rng = np.random.default_rng(1)
centers = _rng.normal(size=(C_, D_)) * 3
maps = _rng.normal(size=(C_, O_, D_)) / 9.0

def toy_batch(n, rng):
    c = rng.integers(0, C_, n)
    x = centers[c] + 0.5 * rng.normal(size=(n, D_))
    y = np.einsum('nod,nd->no', maps[c], x)
    return x, y

class Adam:
    def __init__(self, params, lr):
        self.p, self.lr, self.t = params, lr, 0
        self.m = [np.zeros_like(p) for p in params]; self.v = [np.zeros_like(p) for p in params]
    def step(self, grads):
        self.t += 1
        for p, g, m, v in zip(self.p, grads, self.m, self.v):
            m *= 0.9; m += 0.1 * g; v *= 0.999; v += 0.001 * g * g
            p -= self.lr * (m / (1 - 0.9 ** self.t)) / (np.sqrt(v / (1 - 0.999 ** self.t)) + 1e-8)

In [ ]:
def aux_grad(f, n, alpha, E):
    # TODO: return the gradient of alpha * E * sum_i f_i * P_i with respect to each p_{t,i}.
    #       It's the same for every token: an array of shape (E,).
    raise NotImplementedError

In [ ]:
def train_moe(alpha, seed=1, steps=1500, n=256, bias0=4.0):
    rng = np.random.default_rng(seed)
    Wr = rng.normal(size=(D_, E_)) * 0.1; br = np.zeros(E_); br[0] = bias0
    W = rng.normal(size=(E_, O_, D_)) * 0.1
    opt, drng = Adam([Wr, br, W], 1e-2), np.random.default_rng(123)
    for _ in range(steps):
        x, y = toy_batch(n, drng)
        p = softmax(x @ Wr + br); i = p.argmax(1); pg = p[np.arange(n), i]
        out = np.einsum('nod,nd->no', W[i], x)
        gy = 2 * (pg[:, None] * out - y) / (n * O_)                 # dMSE / dy_hat
        gW = np.zeros_like(W)
        np.add.at(gW, i, pg[:, None, None] * gy[:, :, None] * x[:, None, :])
        dp = np.zeros_like(p); dp[np.arange(n), i] = (gy * out).sum(1)
        if alpha > 0:
            f = np.bincount(i, minlength=E_) / n
            dp += aux_grad(f, n, alpha, E_)                         # your gradient
        dz = p * (dp - (dp * p).sum(1, keepdims=True))              # back through the softmax
        opt.step([x.T @ dz, dz.sum(0), gW])
    x, y = toy_batch(4096, np.random.default_rng(999))
    p = softmax(x @ Wr + br); i = p.argmax(1); pg = p[np.arange(len(x)), i]
    mse = np.mean((pg[:, None] * np.einsum('nod,nd->no', W[i], x) - y) ** 2)
    return mse, np.bincount(i, minlength=E_) / len(x), (Wr, br)

assert np.allclose(aux_grad(np.array([0.5, 0.5]), n=10, alpha=0.1, E=2), [0.01, 0.01])
mse0, load0, _ = train_moe(alpha=0.0)
mse1, load1, _ = train_moe(alpha=0.1)
print(f"no balancing : test MSE {mse0:.4f}, loads {np.round(load0, 2)}")
print(f"aux loss 0.1 : test MSE {mse1:.4f}, loads {np.round(load1, 2)}")
assert load0.max() > 0.4, "without balancing the favourite expert should keep a big share"
assert load1.max() < 0.3, "with the balancing loss no expert should take more than 30%"
assert mse1 < mse0
print("Exercise 4 passed")

## 5. Balancing with a bias instead of a loss (DeepSeek-V3)

DeepSeek-V3 adds a per-expert bias $b_i$ to the scores **only when choosing the top-k**, and after every batch nudges it: $b_i \leftarrow b_i - \gamma$ if expert $i$ got more than its share, $b_i \leftarrow b_i + \gamma$ if less. The gates still use the raw scores.

Write `bias_update(bias, load, gamma)`, where `load` holds each expert's share of assignments for the batch.

In [ ]:
def bias_update(bias, load, gamma):
    # TODO: compare each load with the mean share 1/E and move the bias against the error
    raise NotImplementedError

In [ ]:
assert np.allclose(bias_update(np.zeros(4), np.array([0.4, 0.3, 0.2, 0.1]), 0.01), [-0.01, -0.01, 0.01, 0.01])

# A skewed fixed router: scores = x @ Wr + skew. Only the selection bias changes.
rng = np.random.default_rng(3)
Wr_fixed = rng.normal(size=(D_, E_)) * 0.3
skew = np.array([2.0, 1.0, 0.5, 0.0, 0.0, -0.5, -1.0, -1.5])
bias = np.zeros(E_)
drng = np.random.default_rng(7)
history = []
for step in range(300):
    x, _ = toy_batch(256, drng)
    scores = x @ Wr_fixed + skew
    chosen = np.argsort(-(scores + bias), axis=1)[:, :2]          # top-2 by score + bias
    load = np.bincount(chosen.ravel(), minlength=E_) / chosen.size
    history.append(load.max())
    bias = bias_update(bias, load, gamma=0.01)
print("busiest expert share: first batch", round(history[0], 3), "→ last batch", round(history[-1], 3), "(perfect = 0.125)")
assert history[0] > 0.3 and history[-1] < 0.2
print("Exercise 5 passed")

## Experiment

1. **Capacity factor vs dropped assignments.** Route random batches through a skewed router and sweep the capacity factor. How much padding do you need before drops become rare, with and without balancing?
2. **Collapse is about the start.** Re-run `train_moe(alpha=0.0, bias0=...)` with `bias0` of 0, 2 and 4. When does the router recover on its own?
3. **How big should $\alpha$ be?** Try 0.01, 0.03, 0.3, 1.0. At some point balancing starts to fight the task. Watch the MSE.

In [ ]:
rng = np.random.default_rng(11)
cfs = [0.75, 1.0, 1.25, 1.5, 2.0, 3.0]
skewed, balanced = [], []
for cf in cfs:
    logits_s = rng.normal(size=(1024, 8)) + np.array([1.5, 1.0, 0.5, 0, 0, -0.5, -1, -1.5])
    logits_b = rng.normal(size=(1024, 8))
    for logits, out in ((logits_s, skewed), (logits_b, balanced)):
        _, idx, _ = route(logits, k=2)
        keep, _ = apply_capacity(idx, E=8, cf=cf)
        out.append((~keep).mean())
plt.figure(figsize=(6, 3.5))
plt.plot(cfs, skewed, "o-", label="skewed router")
plt.plot(cfs, balanced, "o-", label="balanced router")
plt.xlabel("capacity factor"); plt.ylabel("share of assignments dropped"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Padding buys fewer drops; balance buys more")
plt.show()

## Solutions

Try the exercises first. Each solution below defines the same function as its stub, so running a solution cell and then the check cell again should pass.

### Solution 1

In [ ]:
def route(logits, k):
    probs = softmax(logits)
    idx = np.argsort(-probs, axis=1)[:, :k]
    chosen = np.take_along_axis(probs, idx, axis=1)
    gates = chosen / chosen.sum(axis=1, keepdims=True)
    return probs, idx, gates

### Solution 2

In [ ]:
def aux_loss(probs, idx):
    T, E = probs.shape
    f = np.bincount(idx.ravel(), minlength=E) / idx.size
    P = probs.mean(axis=0)
    return E * np.sum(f * P)

def z_loss(logits):
    m = logits.max(axis=1, keepdims=True)
    lse = (m + np.log(np.exp(logits - m).sum(axis=1, keepdims=True)))[:, 0]
    return np.mean(lse ** 2)

### Solution 3

In [ ]:
def apply_capacity(idx, E, cf):
    T, k = idx.shape
    C = int(np.ceil(cf * k * T / E))
    keep = np.zeros_like(idx, dtype=bool)
    used = np.zeros(E, dtype=int)
    for t in range(T):
        for j in range(k):
            e = idx[t, j]
            if used[e] < C:
                keep[t, j] = True
                used[e] += 1
    return keep, C

### Solution 4

In [ ]:
def aux_grad(f, n, alpha, E):
    return alpha * E * f / n

### Solution 5

In [ ]:
def bias_update(bias, load, gamma):
    return bias - gamma * np.sign(load - 1.0 / len(load))